# Benchmark Model Evaluator melalui OpenRouter — FINAL V5

Notebook final untuk menetapkan **3 Evaluator Agent** dari **6 kandidat evaluator** pada penelitian penerjemahan Bahasa Duri → Indonesia.

## Dataset final yang dikunci
- master clean corpus: **528 pasangan**
- experimental unique corpus: **523 pasangan**
- train: **417 pasangan**
- validation: **53 pasangan**
- test: **53 pasangan** — **tidak dibaca** oleh notebook benchmark evaluator
- benchmark evaluator: **53 kasus terkontrol**, satu kasus untuk setiap `row_id` validation final

## Prinsip benchmark
1. Benchmark dibentuk dari **validation final**, bukan test.
2. `source_text` dan `reference_text` diverifikasi otomatis terhadap `validation.csv` sebelum API dipanggil.
3. Evaluator **reference-free**: hanya source Duri, candidate translation, shared evidence Top-3 dari train, dan rubrik yang masuk prompt.
4. Reference, gold label, dan metadata konstruksi **tidak masuk prompt evaluator**.
5. Shared evidence hanya dari **417 train**; exact source/reference match dengan kasus aktif dilarang.
6. Tahap A: 53 × 6 × 1 = **318 observasi**.
7. Top-4 masuk Tahap B; rep-1 dari Tahap A digunakan kembali dan ditambah rep-2/rep-3 = **424 panggilan tambahan**.
8. Total observasi benchmark = **742**, ditambah smoke test maksimal 6 panggilan.
9. Metrik utama: **macro-F1 `error_type`**; severity, verdict, joint exact match, validitas JSON/logis, dan stabilitas sebagai pendukung.
10. Panel akhir = **Top-3 performa + stabilitas**, tanpa constraint proprietary/open-weight.
11. Agreement antar-model hanya diagnostik.
12. Versi kandidat dibekukan untuk seluruh benchmark; preflight OpenRouter memverifikasi model dan provider sebelum API benchmark dijalankan.

### Kandidat final
- E1 — GPT-5.2
- E2 — Claude Sonnet 4.6
- E3 — Gemini 3.1 Pro Preview
- E4 — Llama 4 Maverick
- E5 — Qwen3-32B
- E6 — DeepSeek V4 Pro

> Jalankan di Google Colab CPU. GPU tidak diperlukan karena inferensi evaluator dilakukan melalui OpenRouter API.


## 1. Instalasi dependensi

In [ ]:
%pip -q install --upgrade-strategy only-if-needed \
    "sentence-transformers>=3,<6" \
    "openpyxl>=3.1,<4" \
    "tqdm>=4.66,<5"

## 2. Import dan konfigurasi

In [ ]:
from __future__ import annotations
import os, re, json, time, math, random, hashlib, platform, itertools
import importlib.metadata as importlib_metadata
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sentence_transformers import SentenceTransformer
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path('/content/drive/MyDrive/10122233_CikitaMeydilvira_Skripsi')

# FINAL dataset paths — jangan gunakan fallback ke dataset versi lama.
DATASET_CANDIDATES = [
    PROJECT_ROOT / 'Dataset' / 'benchmark_evaluator.csv',
]
TRAIN_CANDIDATES = [
    PROJECT_ROOT / 'Dataset' / '07_dataset_split_final' / 'train.csv',
]
VALIDATION_CANDIDATES = [
    PROJECT_ROOT / 'Dataset' / '07_dataset_split_final' / 'validation.csv',
]

OUTPUT_ROOT = PROJECT_ROOT / 'Hasil' / 'benchmark_evaluator_openrouter_FINAL_V5'

EXPECTED_CASES = 53
EXPECTED_TRAIN_ROWS = 417
EXPECTED_VALIDATION_ROWS = 53

OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
OPENROUTER_CHAT_URL = f'{OPENROUTER_BASE_URL}/chat/completions'
OPENROUTER_MODELS_URL = f'{OPENROUTER_BASE_URL}/models'
OPENROUTER_PROVIDERS_URL = f'{OPENROUTER_BASE_URL}/providers'
APP_TITLE = 'Skripsi Benchmark Evaluator Bahasa Duri'
HTTP_REFERER = ''

# FINAL candidate set — freeze selama seluruh benchmark.
MODELS = {
    'E1': {
        'display_name':'GPT-5.2',
        'model_id':'openai/gpt-5.2',
        'developer':'OpenAI',
        'family':'GPT',
        'access_type':'Proprietary',
        'preferred':['openai'],
    },
    'E2': {
        'display_name':'Claude Sonnet 4.6',
        'model_id':'anthropic/claude-sonnet-4.6',
        'developer':'Anthropic',
        'family':'Claude',
        'access_type':'Proprietary',
        'preferred':['anthropic'],
    },
    'E3': {
        'display_name':'Gemini 3.1 Pro Preview',
        'model_id':'google/gemini-3.1-pro-preview',
        'developer':'Google',
        'family':'Gemini',
        'access_type':'Proprietary',
        'preferred':['google-ai-studio','google-vertex'],
    },
    'E4': {
        'display_name':'Llama 4 Maverick',
        'model_id':'meta-llama/llama-4-maverick',
        'developer':'Meta',
        'family':'Llama',
        'access_type':'Open-weight',
        'preferred':['deepinfra','google-vertex','novita'],
    },
    'E5': {
        'display_name':'Qwen3-32B',
        'model_id':'qwen/qwen3-32b',
        'developer':'Qwen/Alibaba',
        'family':'Qwen',
        'access_type':'Open-weight',
        'preferred':['deepinfra','nebius','siliconflow','groq'],
    },
    'E6': {
        'display_name':'DeepSeek V4 Pro',
        'model_id':'deepseek/deepseek-v4-pro',
        'developer':'DeepSeek',
        'family':'DeepSeek',
        'access_type':'Open-weight',
        'preferred':['novita','deepinfra','siliconflow','deepseek'],
    },
}

ERROR_TYPES = [
    'correct','meaning_shift','missing_information','added_information',
    'hallucination','untranslated','empty_output','fluency_issue','source_leakage'
]
SEVERITIES = ['none','minor','major','critical']
VERDICTS = ['layak','perlu_ditinjau']

SHORTLIST_SIZE = 4
TOTAL_REPETITIONS_SHORTLIST = 3
MAX_TOKENS = 2048
REQUESTED_TEMPERATURE = 0.0

RUN_SMOKE_TEST = True
RUN_STAGE_A = True
RUN_STAGE_B = True
SMOKE_CASE_ID = 'C001'

PIN_PROVIDER = True
ALLOW_PROVIDER_FALLBACKS = False
REQUIRE_REQUEST_PARAMETERS = True
DATA_COLLECTION_POLICY = 'allow'

REQUEST_TIMEOUT_SECONDS = 300
MAX_HTTP_RETRIES = 6
BASE_RETRY_DELAY_SECONDS = 2.0
SLEEP_BETWEEN_CALLS_SECONDS = 0.20

RETRIEVAL_MODEL_ID = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
RETRIEVAL_TOP_K = 3
RETRIEVAL_DEVICE = 'cpu'

N_MODELS = len(MODELS)
STAGE_A_EXPECTED = EXPECTED_CASES * N_MODELS
STAGE_B_ADDITIONAL_EXPECTED = (
    EXPECTED_CASES
    * SHORTLIST_SIZE
    * (TOTAL_REPETITIONS_SHORTLIST - 1)
)
TOTAL_BENCHMARK_OBSERVATIONS = (
    STAGE_A_EXPECTED + STAGE_B_ADDITIONAL_EXPECTED
)
MAX_API_CALLS_WITH_SMOKE = (
    TOTAL_BENCHMARK_OBSERVATIONS
    + (N_MODELS if RUN_SMOKE_TEST else 0)
)

assert N_MODELS == 6
assert SHORTLIST_SIZE == 4
assert TOTAL_REPETITIONS_SHORTLIST == 3

print('Kandidat evaluator:', N_MODELS)
for k, v in MODELS.items():
    print(
        k, v['display_name'],
        '|', v['developer'],
        '|', v['access_type'],
        '->', v['model_id']
    )

print('Stage A:', STAGE_A_EXPECTED)
print('Stage B tambahan:', STAGE_B_ADDITIONAL_EXPECTED)
print('Total observasi benchmark:', TOTAL_BENCHMARK_OBSERVATIONS)
print('Maksimum API call termasuk smoke:', MAX_API_CALLS_WITH_SMOKE)


Kandidat evaluator: 6
E1 GPT-5.2 | OpenAI | Proprietary -> openai/gpt-5.2
E2 Claude Sonnet 4.6 | Anthropic | Proprietary -> anthropic/claude-sonnet-4.6
E3 Gemini 3.1 Pro Preview | Google | Proprietary -> google/gemini-3.1-pro-preview
E4 Llama 4 Maverick | Meta | Open-weight -> meta-llama/llama-4-maverick
E5 Qwen3-32B | Qwen/Alibaba | Open-weight -> qwen/qwen3-32b
E6 DeepSeek V4 Pro | DeepSeek | Open-weight -> deepseek/deepseek-v4-pro
Stage A: 318
Stage B tambahan: 424
Total observasi benchmark: 742
Maksimum API call termasuk smoke: 748


## 3. Mount Google Drive dan baca OpenRouter Secret

In [ ]:
try:
    from google.colab import drive, userdata
except ImportError as exc:
    raise RuntimeError('Notebook ini ditujukan untuk Google Colab.') from exc

drive.mount('/content/drive', force_remount=False)
try:
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception as exc:
    raise RuntimeError('Colab Secret OPENROUTER_API_KEY tidak dapat dibaca.') from exc
if not OPENROUTER_API_KEY or not str(OPENROUTER_API_KEY).strip():
    raise RuntimeError('OPENROUTER_API_KEY kosong. Tambahkan pada Colab Secrets dan aktifkan Notebook access.')
print('Google Drive OK; OpenRouter key terdeteksi (tidak ditampilkan).')

Mounted at /content/drive
Google Drive OK; OpenRouter key terdeteksi (tidak ditampilkan).


## 4. Load dan validasi dataset

Notebook memuat:
1. `Dataset/benchmark_evaluator.csv` — **53 kasus benchmark final**,
2. `Dataset/07_dataset_split_final/train.csv` — **417 pasangan train final**,
3. `Dataset/07_dataset_split_final/validation.csv` — **53 pasangan validation final**, hanya untuk integrity gate.

Sebelum panggilan API, notebook memastikan bahwa membership 53 kasus benchmark sama dengan validation final serta `source_text` dan `reference_text` cocok **persis**. Test set tidak dibaca.


In [ ]:
def resolve_existing_path(candidates,label):
    for p in candidates:
        if p.exists(): return p
    raise FileNotFoundError(f'{label} tidak ditemukan.\nDicari di:\n'+'\n'.join(map(str,candidates)))
DATASET_PATH=resolve_existing_path(DATASET_CANDIDATES,'dataset benchmark')
TRAIN_PATH=resolve_existing_path(TRAIN_CANDIDATES,'train.csv final')
VALIDATION_PATH=resolve_existing_path(VALIDATION_CANDIDATES,'validation.csv final')
df_gold_full=pd.read_csv(DATASET_PATH,dtype=str,keep_default_na=False,encoding='utf-8-sig')
df_train=pd.read_csv(TRAIN_PATH,dtype=str,keep_default_na=False,encoding='utf-8-sig')
df_validation=pd.read_csv(VALIDATION_PATH,dtype=str,keep_default_na=False,encoding='utf-8-sig')
for df in [df_gold_full,df_train,df_validation]: df.columns=[str(c).strip() for c in df.columns]
REQ_GOLD=['case_id','source_split','source_row_id','source_text','reference_text','candidate_text','gold_error_type','gold_severity','gold_verdict','construction_method','construction_note']
REQ_TRAIN=['row_id','conversation_no','turn_order','duri_text','indonesia_text']
REQ_VALID=['row_id','conversation_no','turn_order','duri_text','indonesia_text']
for label,df,req in [('benchmark',df_gold_full,REQ_GOLD),('train',df_train,REQ_TRAIN),('validation',df_validation,REQ_VALID)]:
    missing=[c for c in req if c not in df.columns]
    if missing: raise ValueError(f'Kolom {label} kurang: '+', '.join(missing))
if len(df_gold_full)!=EXPECTED_CASES: raise ValueError(f'Benchmark harus {EXPECTED_CASES} kasus, ditemukan {len(df_gold_full)}.')
if len(df_train)!=EXPECTED_TRAIN_ROWS: raise ValueError(f'Train final harus {EXPECTED_TRAIN_ROWS} pasangan, ditemukan {len(df_train)}. Jangan gunakan versi dataset lama.')
if len(df_validation)!=EXPECTED_VALIDATION_ROWS: raise ValueError(f'Validation final harus {EXPECTED_VALIDATION_ROWS} pasangan, ditemukan {len(df_validation)}.')
if df_gold_full['case_id'].duplicated().any(): raise ValueError('case_id benchmark duplikat.')
if df_gold_full['source_row_id'].duplicated().any(): raise ValueError('source_row_id benchmark duplikat.')
if df_validation['row_id'].duplicated().any(): raise ValueError('row_id validation duplikat.')
expected_case_ids=[f'C{i:03d}' for i in range(1,EXPECTED_CASES+1)]
if df_gold_full['case_id'].tolist()!=expected_case_ids: raise ValueError('Urutan case_id harus tepat C001 ... C053.')
if not df_gold_full['source_split'].str.strip().str.lower().eq('validation').all(): raise ValueError('Semua source_split benchmark harus validation.')
if set(df_gold_full['gold_error_type'])-set(ERROR_TYPES): raise ValueError('Ada gold_error_type tidak valid.')
if set(df_gold_full['gold_severity'])-set(SEVERITIES): raise ValueError('Ada gold_severity tidak valid.')
if set(df_gold_full['gold_verdict'])-set(VERDICTS): raise ValueError('Ada gold_verdict tidak valid.')
bad_correct=df_gold_full[df_gold_full['gold_error_type'].eq('correct')&~(df_gold_full['gold_severity'].eq('none')&df_gold_full['gold_verdict'].eq('layak'))]
if len(bad_correct): raise ValueError('Label correct harus severity=none dan verdict=layak.')
bad_noncorrect_none=df_gold_full[~df_gold_full['gold_error_type'].eq('correct')&df_gold_full['gold_severity'].eq('none')]
if len(bad_noncorrect_none): raise ValueError('Kasus selain correct tidak boleh severity=none.')
bad_major_critical=df_gold_full[df_gold_full['gold_severity'].isin(['major','critical'])&~df_gold_full['gold_verdict'].eq('perlu_ditinjau')]
if len(bad_major_critical): raise ValueError('Severity major/critical harus verdict=perlu_ditinjau.')
empty_mask=df_gold_full['gold_error_type'].eq('empty_output')
if not df_gold_full.loc[empty_mask,'candidate_text'].str.strip().eq('').all(): raise ValueError('Kasus empty_output harus candidate_text kosong.')
if df_gold_full.loc[~empty_mask,'candidate_text'].str.strip().eq('').any(): raise ValueError('Hanya kasus empty_output yang boleh candidate_text kosong.')
valid_lookup=df_validation.set_index('row_id'); bench_row_ids=df_gold_full['source_row_id'].tolist()
if set(bench_row_ids)!=set(df_validation['row_id']):
    raise ValueError('Membership benchmark tidak sama dengan validation final.')
sync_rows=[]
for _,r in df_gold_full.iterrows():
    v=valid_lookup.loc[r['source_row_id']]
    source_same=str(r['source_text'])==str(v['duri_text']); ref_same=str(r['reference_text'])==str(v['indonesia_text'])
    conv_same=('conversation_no' not in df_gold_full.columns or str(r.get('conversation_no',''))==str(v['conversation_no']))
    turn_same=('turn_order' not in df_gold_full.columns or str(r.get('turn_order',''))==str(v['turn_order']))
    sync_rows.append({'case_id':r['case_id'],'source_row_id':r['source_row_id'],'source_exact_match':source_same,'reference_exact_match':ref_same,'conversation_match':conv_same,'turn_order_match':turn_same})
df_validation_sync=pd.DataFrame(sync_rows)
if not df_validation_sync[['source_exact_match','reference_exact_match','conversation_match','turn_order_match']].all(axis=None):
    display(df_validation_sync[~df_validation_sync[['source_exact_match','reference_exact_match','conversation_match','turn_order_match']].all(axis=1)])
    raise ValueError('Benchmark belum sinkron dengan validation final.')
df_gold=df_gold_full[['case_id','source_text','reference_text','candidate_text','gold_error_type','gold_severity','gold_verdict']].copy().reset_index(drop=True)
print('Benchmark :',DATASET_PATH,'|',len(df_gold_full),'kasus')
print('Train     :',TRAIN_PATH,'|',len(df_train),'pasangan')
print('Validation:',VALIDATION_PATH,'|',len(df_validation),'pasangan')
print('Validation sync: 53/53 source + reference + membership MATCH')
display(df_gold_full['gold_error_type'].value_counts().rename_axis('error_type').to_frame('count'))


Benchmark : /content/drive/MyDrive/10122233_CikitaMeydilvira_Skripsi/Dataset/benchmark_evaluator.csv | 53 kasus
Train     : /content/drive/MyDrive/10122233_CikitaMeydilvira_Skripsi/Dataset/07_dataset_split_final/train.csv | 417 pasangan
Validation: /content/drive/MyDrive/10122233_CikitaMeydilvira_Skripsi/Dataset/07_dataset_split_final/validation.csv | 53 pasangan
Validation sync: 53/53 source + reference + membership MATCH


,count
error_type,
correct,6
untranslated,6
missing_information,6
hallucination,6
source_leakage,6
added_information,6
meaning_shift,6
fluency_issue,6
empty_output,5


## 5. Shared evidence retrieval dari train corpus

Shared evidence Top-3 berasal dari **417 train final** menggunakan `paraphrase-multilingual-mpnet-base-v2`. Exact source match dan exact reference match dengan kasus aktif dilarang. Mekanisme ini khusus evaluator dan berbeda dari retrieval Fragment-Shot Prompting.


In [ ]:
def norm_text(x): return re.sub(r'\s+',' ',str(x or '').strip().lower())
retriever=SentenceTransformer(RETRIEVAL_MODEL_ID,device=RETRIEVAL_DEVICE)
train_emb=retriever.encode(df_train['duri_text'].tolist(),batch_size=64,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
case_emb=retriever.encode(df_gold['source_text'].tolist(),batch_size=64,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
evidence_records=[]; evidence_by_case={}
for i,row in df_gold.iterrows():
    scores=train_emb@case_emb[i]; ranked=np.argsort(-scores); src_norm=norm_text(row['source_text']); ref_norm=norm_text(row['reference_text']); chosen=[]
    for j in ranked:
        tr=df_train.iloc[int(j)]; d=str(tr['duri_text']).strip(); ind=str(tr['indonesia_text']).strip()
        if norm_text(d)==src_norm or norm_text(ind)==ref_norm: continue
        chosen.append({'rank':len(chosen)+1,'train_index':int(j),'train_row_id':str(tr.get('row_id','')),'duri_text':d,'indonesia_text':ind,'similarity':float(scores[int(j)])})
        if len(chosen)==RETRIEVAL_TOP_K: break
    if len(chosen)!=RETRIEVAL_TOP_K: raise RuntimeError(f'Evidence kurang untuk {row["case_id"]}.')
    lines=[]
    for item in chosen:
        lines.append(f"Contoh {item['rank']}:\n- Duri: {item['duri_text']}\n- Indonesia: {item['indonesia_text']}")
        evidence_records.append({'case_id':row['case_id'],**item})
    evidence_by_case[row['case_id']]='\n\n'.join(lines)
df_evidence=pd.DataFrame(evidence_records)
df_blind=df_gold[['case_id','source_text','candidate_text']].copy(); df_blind['shared_evidence']=df_blind['case_id'].map(evidence_by_case)
for forbidden in ['reference_text','gold_error_type','gold_severity','gold_verdict','construction_method','construction_note']: assert forbidden not in df_blind.columns
for _,r in df_blind.iterrows():
    ref=norm_text(df_gold.loc[df_gold['case_id'].eq(r['case_id']),'reference_text'].iloc[0])
    if ref and ref in norm_text(r['shared_evidence']): raise RuntimeError(f'Reference literal bocor ke evidence: {r["case_id"]}')
del retriever,train_emb,case_emb
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
df_blind.to_csv(OUTPUT_ROOT/'blind_input_with_shared_evidence.csv',index=False,encoding='utf-8-sig')
df_evidence.to_csv(OUTPUT_ROOT/'shared_evidence_detail.csv',index=False,encoding='utf-8-sig')
df_validation_sync.to_csv(OUTPUT_ROOT/'benchmark_validation_sync_audit.csv',index=False,encoding='utf-8-sig')
print('Blind input dan shared evidence siap. Reference/gold/construction metadata TIDAK masuk evaluator input.')
display(df_blind.head(2))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Blind input dan shared evidence siap. Reference/gold/construction metadata TIDAK masuk evaluator input.


,case_id,source_text,candidate_text,shared_evidence
0,C001,Indara sangantaq?,Siapa nama Anda?,Contoh 1:\n- Duri: Indara mangbuah jio baraqba...
1,C002,Disanganaq Nur.,Saya disanganaq Nur.,Contoh 1:\n- Duri: Maneqnaq melajah mangbicara...


## 6. Rubrik evaluator dan prompt reference-free

In [ ]:
SYSTEM_PROMPT = """
Anda adalah Evaluator Agent untuk menilai kualitas terjemahan Bahasa Duri ke Bahasa Indonesia.

INPUT YANG BOLEH DIGUNAKAN
1. Source Bahasa Duri.
2. Shared evidence berupa pasangan Duri-Indonesia yang diambil dari korpus latih.
3. Candidate translation.

Shared evidence adalah bukti pendukung, BUKAN reference/gold untuk kasus yang sedang dinilai. Anda tidak menerima reference Indonesia atau gold label kasus.
Pilih tepat satu error_type yang paling dominan.

ERROR_TYPE
- correct: makna dan informasi penting terjaga, bahasa Indonesia dapat diterima, tidak ada kesalahan dominan.
- meaning_shift: makna utama bergeser, berubah, terbalik, atau hubungan semantis penting berubah.
- missing_information: informasi penting dari source hilang/tidak diterjemahkan, sementara bagian tersisa masih berupa terjemahan Indonesia.
- added_information: ada informasi tambahan yang tidak didukung, tetapi belum merupakan halusinasi berat yang mengganti inti makna.
- hallucination: isi substansial tidak didukung source/evidence, sangat tidak berkaitan, atau mengganti makna utama dengan informasi rekaan.
- untranslated: sebagian/seluruh unsur Bahasa Duri yang seharusnya diterjemahkan tetap menjadi isi kandidat.
- empty_output: kandidat kosong atau tidak menghasilkan terjemahan yang dapat dinilai.
- fluency_issue: makna utama relatif terjaga, tetapi bahasa Indonesia tidak alami, tidak gramatikal, sangat kaku, atau sulit dibaca.
- source_leakage: terjemahan Indonesia tersedia, tetapi source Bahasa Duri/instruksi/prompt ikut tercetak.

SEVERITY
- none: hanya untuk correct.
- minor: kesalahan terbatas dan tidak mengubah informasi utama secara substantif.
- major: kesalahan memengaruhi informasi penting atau kegunaan terjemahan.
- critical: keluaran kosong/tidak dapat digunakan, halusinasi berat, atau perubahan makna sangat serius.

VERDICT
- layak: dapat digunakan tanpa perbaikan substantif.
- perlu_ditinjau: perlu pemeriksaan/perbaikan sebelum digunakan.

ATURAN KONSISTENSI
- correct -> severity=none dan verdict=layak.
- selain correct -> severity tidak boleh none.
- severity major/critical -> verdict=perlu_ditinjau.
- empty_output -> severity=critical dan verdict=perlu_ditinjau.

BATASAN
- Jangan memperbaiki/menulis ulang kandidat.
- Jangan memberi skor numerik.
- Jangan menebak apakah kandidat berasal dari Zero-Shot, FSP, atau LoRA.
- Jangan menggunakan reference/gold yang tidak diberikan.
- reason maksimal dua kalimat.

Keluarkan HANYA satu object JSON tanpa markdown/code fence/teks tambahan:
{
  "error_type": "correct|meaning_shift|missing_information|added_information|hallucination|untranslated|empty_output|fluency_issue|source_leakage",
  "severity": "none|minor|major|critical",
  "verdict": "layak|perlu_ditinjau",
  "reason": "alasan singkat"
}
""".strip()

USER_PROMPT_TEMPLATE = """
Nilai kasus berikut.

<case_id>{case_id}</case_id>
<source_duri>{source_text}</source_duri>
<shared_evidence>{shared_evidence}</shared_evidence>
<candidate_translation>{candidate_text}</candidate_translation>
""".strip()

def build_user_prompt(row):
    candidate=str(row['candidate_text'])
    return USER_PROMPT_TEMPLATE.format(case_id=row['case_id'],source_text=row['source_text'],shared_evidence=row['shared_evidence'],candidate_text=candidate if candidate.strip() else '[KOSONG]')

PROMPT_HASH=hashlib.sha256((SYSTEM_PROMPT+'\n'+USER_PROMPT_TEMPLATE).encode()).hexdigest()[:16]
print('Prompt hash:',PROMPT_HASH)
print(build_user_prompt(df_blind.iloc[0]))

Prompt hash: d0e3d1938a7dcd05
Nilai kasus berikut.

<case_id>C001</case_id>
<source_duri>Indara sangantaq?</source_duri>
<shared_evidence>Contoh 1:
- Duri: Indara mangbuah jio baraqbah?
- Indonesia: Siapakah yang mengolah tanah di kebun?

Contoh 2:
- Duri: Indara nataqbangan punti?
- Indonesia: Untuk siapa dia menebang pisang?

Contoh 3:
- Duri: Indara nnampai pea indeq?
- Indonesia: Siapakah yang menjaga anak di sini?</shared_evidence>
<candidate_translation>Siapa nama Anda?</candidate_translation>


## 7. Helper OpenRouter HTTP

In [ ]:
class FatalAPIError(RuntimeError): pass
class TransientAPIError(RuntimeError): pass

def utc_now_iso(): return datetime.now(timezone.utc).isoformat()
def headers():
    h={'Authorization':f'Bearer {OPENROUTER_API_KEY}','Content-Type':'application/json','X-Title':APP_TITLE}
    if HTTP_REFERER.strip(): h['HTTP-Referer']=HTTP_REFERER.strip()
    return h

def request_retry(method,url,payload=None):
    retry_status={408,409,425,429,500,502,503,504}; last=''
    for attempt in range(1,MAX_HTTP_RETRIES+1):
        try:
            r=requests.request(method,url,headers=headers(),json=payload,timeout=REQUEST_TIMEOUT_SECONDS)
            if r.status_code<400: return r
            last=f'HTTP {r.status_code}: {r.text[:3000]}'
            if r.status_code not in retry_status: raise FatalAPIError(last)
        except requests.RequestException as exc:
            last=f'{type(exc).__name__}: {exc}'
        if attempt<MAX_HTTP_RETRIES:
            time.sleep(min(BASE_RETRY_DELAY_SECONDS*(2**(attempt-1)),60.0))
    raise TransientAPIError(last or 'OpenRouter request gagal.')

def get_json(url):
    r=request_retry('GET',url)
    try:return r.json()
    except ValueError as exc: raise FatalAPIError('GET response bukan JSON: '+url) from exc

## 8. Preflight model dan provider pinning

In [ ]:
catalog=get_json(OPENROUTER_MODELS_URL).get('data') or []
catalog_by_id={x.get('id'):x for x in catalog if x.get('id')}
providers=get_json(OPENROUTER_PROVIDERS_URL).get('data') or []
name_to_slug={str(x.get('name','')).strip().lower():str(x.get('slug','')).strip().lower() for x in providers if x.get('name') and x.get('slug')}

def endpoints_url(model_id):
    a,s=model_id.split('/',1); return f'{OPENROUTER_BASE_URL}/models/{a}/{s}/endpoints'
def provider_slug(ep):
    direct=ep.get('provider_slug') or ep.get('provider') or ep.get('tag')
    if direct:return str(direct).strip().lower()
    name=str(ep.get('provider_name') or '').strip()
    if name:return name_to_slug.get(name.lower(),re.sub(r'[^a-z0-9]+','-',name.lower()).strip('-'))
    return ''
def choose_provider(info,payload):
    data=payload.get('data') or {}; eps=data.get('endpoints') or payload.get('endpoints') or []
    vals=[(provider_slug(ep),ep) for ep in eps if provider_slug(ep)]
    if not vals: raise RuntimeError('Tidak ada endpoint untuk '+info['model_id'])
    for pref in info['preferred']:
        matches=[x for x in vals if x[0]==pref or x[0].startswith(pref+'/')]
        if matches:return sorted(matches,key=lambda x:x[0])[0]
    return sorted(vals,key=lambda x:x[0])[0]

SELECTED_PROVIDERS={}; SUPPORTED={}; pf=[]
for key,info in MODELS.items():
    if info['model_id'] not in catalog_by_id:
        raise RuntimeError('Model tidak ada di katalog OpenRouter: '+info['model_id'])
    meta=catalog_by_id[info['model_id']]
    SUPPORTED[key]=set(meta.get('supported_parameters') or [])
    slug,ep=choose_provider(info,get_json(endpoints_url(info['model_id'])))
    SELECTED_PROVIDERS[key]=slug
    pf.append({
        'evaluator_key':key,
        'display_name':info['display_name'],
        'developer':info['developer'],
        'family':info['family'],
        'access_type':info['access_type'],
        'model_id':info['model_id'],
        'selected_provider':slug,
        'supported_parameters':', '.join(sorted(SUPPORTED[key])),
        'endpoint_quantization':ep.get('quantization',''),
        'endpoint_context_length':ep.get('context_length',''),
        'canonical_slug':meta.get('canonical_slug','')
    })

df_preflight=pd.DataFrame(pf)
SEND_TEMPERATURE=all('temperature' in SUPPORTED[k] for k in MODELS)
print('Provider dipatok per model; fallback=False.')
display(df_preflight)
print('Temperature:', REQUESTED_TEMPERATURE if SEND_TEMPERATURE else 'tidak dikirim karena tidak didukung seragam semua kandidat')

Provider dipatok per model; fallback=False.


,evaluator_key,display_name,developer,family,access_type,model_id,selected_provider,supported_parameters,endpoint_quantization,endpoint_context_length,canonical_slug
0,E1,GPT-5.2,OpenAI,GPT,Proprietary,openai/gpt-5.2,openai,"include_reasoning, max_completion_tokens, max_...",unknown,400000,openai/gpt-5.2-20251211
1,E2,Claude Sonnet 4.6,Anthropic,Claude,Proprietary,anthropic/claude-sonnet-4.6,anthropic,"include_reasoning, max_completion_tokens, max_...",unknown,1000000,anthropic/claude-4.6-sonnet-20260217
2,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,google/gemini-3.1-pro-preview,google-ai-studio,"include_reasoning, max_tokens, reasoning, reas...",unknown,1048576,google/gemini-3.1-pro-preview-20260219
3,E4,Llama 4 Maverick,Meta,Llama,Open-weight,meta-llama/llama-4-maverick,deepinfra/base,"frequency_penalty, logit_bias, logprobs, max_t...",fp8,1048576,meta-llama/llama-4-maverick-17b-128e-instruct
4,E5,Qwen3-32B,Qwen/Alibaba,Qwen,Open-weight,qwen/qwen3-32b,deepinfra/fp8,"frequency_penalty, include_reasoning, logit_bi...",fp8,40960,qwen/qwen3-32b-04-28
5,E6,DeepSeek V4 Pro,DeepSeek,DeepSeek,Open-weight,deepseek/deepseek-v4-pro,novita/fp8,"frequency_penalty, include_reasoning, logit_bi...",fp8,1048576,deepseek/deepseek-v4-pro-20260423


Temperature: tidak dikirim karena tidak didukung seragam semua kandidat


## 9. RUN_ID dan output

In [ ]:
def sha_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for ch in iter(lambda:f.read(1024*1024),b''): h.update(ch)
    return h.hexdigest()[:16]
DATASET_HASH=sha_file(DATASET_PATH); TRAIN_HASH=sha_file(TRAIN_PATH); VALIDATION_HASH=sha_file(VALIDATION_PATH)
run_payload={'dataset':DATASET_HASH,'train':TRAIN_HASH,'validation':VALIDATION_HASH,'prompt':PROMPT_HASH,'models':{k:v['model_id'] for k,v in MODELS.items()},'providers':SELECTED_PROVIDERS,'retrieval':RETRIEVAL_MODEL_ID,'topk':RETRIEVAL_TOP_K,'max_tokens':MAX_TOKENS,'send_temperature':SEND_TEMPERATURE,'selection_policy':'capability_then_stability_no_access_type_constraint'}
RUN_ID=hashlib.sha256(json.dumps(run_payload,sort_keys=True).encode()).hexdigest()[:16]
RUN_DIR=OUTPUT_ROOT/f'run_{RUN_ID}'; RUN_DIR.mkdir(parents=True,exist_ok=True)
ATTEMPTS_PATH=RUN_DIR/'benchmark_attempts.csv'; STAGE_A_PATH=RUN_DIR/'stage_a_capability_summary.csv'; SHORTLIST_PATH=RUN_DIR/'stage_a_shortlist_4.csv'; STAGE_B_PATH=RUN_DIR/'stage_b_performance_stability_summary.csv'; PANEL_COMB_PATH=RUN_DIR/'panel_combinations_diagnostic.csv'; PANEL_TOP3_PATH=RUN_DIR/'panel_top3_final_recommendation.csv'; PANEL_CASE_PATH=RUN_DIR/'panel_top3_case_predictions.csv'; REPORT_XLSX=RUN_DIR/'benchmark_evaluator_report.xlsx'
df_preflight.to_csv(RUN_DIR/'openrouter_preflight.csv',index=False,encoding='utf-8-sig'); df_blind.to_csv(RUN_DIR/'blind_input_with_shared_evidence.csv',index=False,encoding='utf-8-sig'); df_evidence.to_csv(RUN_DIR/'shared_evidence_detail.csv',index=False,encoding='utf-8-sig'); df_validation_sync.to_csv(RUN_DIR/'benchmark_validation_sync_audit.csv',index=False,encoding='utf-8-sig'); df_gold_full.to_csv(RUN_DIR/'benchmark_gold_audit_copy.csv',index=False,encoding='utf-8-sig')
with open(RUN_DIR/'run_config.json','w',encoding='utf-8') as f:
    json.dump({'created_at_utc':utc_now_iso(),'run_id':RUN_ID,'dataset_path':str(DATASET_PATH),'dataset_hash':DATASET_HASH,'train_path':str(TRAIN_PATH),'train_hash':TRAIN_HASH,'validation_path':str(VALIDATION_PATH),'validation_hash':VALIDATION_HASH,'validation_sync_all_exact':True,'prompt_hash':PROMPT_HASH,'models':MODELS,'selected_providers':SELECTED_PROVIDERS,'temperature_sent':SEND_TEMPERATURE,'requested_temperature':REQUESTED_TEMPERATURE,'max_tokens':MAX_TOKENS,'retrieval_model':RETRIEVAL_MODEL_ID,'retrieval_top_k':RETRIEVAL_TOP_K,'expected_calls':{'smoke':N_MODELS if RUN_SMOKE_TEST else 0,'stage_a':STAGE_A_EXPECTED,'stage_b_additional':STAGE_B_ADDITIONAL_EXPECTED,'benchmark_observations':TOTAL_BENCHMARK_OBSERVATIONS,'max_api_calls_including_smoke':MAX_API_CALLS_WITH_SMOKE},'selection_policy':'Macro-F1 error_type primary; stability/supporting metrics secondary; family/access type descriptive only; inter-model agreement diagnostic only.','versions':{'python':platform.python_version(),'pandas':importlib_metadata.version('pandas'),'numpy':importlib_metadata.version('numpy'),'requests':importlib_metadata.version('requests'),'sklearn':importlib_metadata.version('scikit-learn'),'sentence_transformers':importlib_metadata.version('sentence-transformers')}},f,ensure_ascii=False,indent=2)
with open(RUN_DIR/'evaluator_prompt.txt','w',encoding='utf-8') as f: f.write(SYSTEM_PROMPT+'\n\n'+USER_PROMPT_TEMPLATE)
print('RUN_ID:',RUN_ID); print('RUN_DIR:',RUN_DIR)


RUN_ID: 372a575a9fa21f91
RUN_DIR: /content/drive/MyDrive/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_evaluator_openrouter_FINAL_V5/run_372a575a9fa21f91


## 10. Parse + validasi JSON evaluator

In [ ]:
def parse_json_object(text):
    raw=str(text or '').strip().lstrip('\ufeff')
    if not raw: raise ValueError('Respons kosong.')
    try:
        x=json.loads(raw)
        if not isinstance(x,dict):raise ValueError('JSON harus object.')
        return x,True,raw
    except json.JSONDecodeError:pass
    dec=json.JSONDecoder()
    for m in re.finditer(r'\{',raw):
        try:x,end=dec.raw_decode(raw[m.start():])
        except json.JSONDecodeError:continue
        if isinstance(x,dict):return x,False,raw[m.start():m.start()+end]
    raise ValueError('Object JSON tidak ditemukan.')

def validate_prediction(p):
    schema=[]; logical=[]; req={'error_type','severity','verdict','reason'}
    if not isinstance(p,dict):return False,False,['Bukan JSON object.']
    if req-set(p):schema.append('Atribut hilang: '+str(sorted(req-set(p))))
    if set(p)-req:schema.append('Atribut tambahan: '+str(sorted(set(p)-req)))
    if p.get('error_type') not in ERROR_TYPES:schema.append('error_type invalid')
    if p.get('severity') not in SEVERITIES:schema.append('severity invalid')
    if p.get('verdict') not in VERDICTS:schema.append('verdict invalid')
    if not isinstance(p.get('reason'),str) or not p.get('reason','').strip():schema.append('reason kosong/invalid')
    schema_ok=not schema
    if schema_ok:
        e,s,v=p['error_type'],p['severity'],p['verdict']
        if e=='correct' and (s!='none' or v!='layak'):logical.append('correct harus none+layak')
        if e!='correct' and s=='none':logical.append('non-correct tidak boleh none')
        if s in {'major','critical'} and v!='perlu_ditinjau':logical.append('major/critical harus perlu_ditinjau')
        if e=='empty_output' and (s!='critical' or v!='perlu_ditinjau'):logical.append('empty_output harus critical+perlu_ditinjau')
    return schema_ok, schema_ok and not logical, schema+logical

## 11. OpenRouter call + checkpoint

In [ ]:
RESULT_COLUMNS=['timestamp_utc','run_id','stage','evaluator_key','display_name','developer','family','access_type','requested_model','requested_provider','returned_model','returned_provider','case_id','repetition','temperature_sent','max_tokens','api_success','raw_response','extracted_json','pred_error_type','pred_severity','pred_verdict','pred_reason','json_strict_valid','json_extractable','schema_valid','logical_valid','validation_errors','finish_reason','latency_ms','prompt_tokens','completion_tokens','reasoning_tokens','total_tokens','reported_cost','response_id','error_class','error_message']

def normalize_content(c):
    if isinstance(c,str):return c
    if isinstance(c,list):
        out=[]
        for x in c:
            if isinstance(x,str):out.append(x)
            elif isinstance(x,dict) and (x.get('text') or x.get('content')):out.append(str(x.get('text') or x.get('content')))
        return '\n'.join(out).strip()
    return str(c or '').strip()

def make_payload(key,info,row):
    p={
        'model':info['model_id'],
        'messages':[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':build_user_prompt(row)}],
        'max_tokens':MAX_TOKENS,
        'stream':False
    }
    if SEND_TEMPERATURE:p['temperature']=REQUESTED_TEMPERATURE
    if PIN_PROVIDER:
        p['provider']={
            'only':[SELECTED_PROVIDERS[key]],
            'allow_fallbacks':ALLOW_PROVIDER_FALLBACKS,
            'require_parameters':REQUIRE_REQUEST_PARAMETERS,
            'data_collection':DATA_COLLECTION_POLICY
        }
    return p

def call_openrouter(key,info,row):
    t=time.perf_counter(); r=request_retry('POST',OPENROUTER_CHAT_URL,make_payload(key,info,row)); latency=(time.perf_counter()-t)*1000
    data=r.json(); choices=data.get('choices') or []
    if not choices:raise TransientAPIError('Respons tidak punya choices: '+str(data)[:1000])
    ch=choices[0]; msg=ch.get('message') or {}; usage=data.get('usage') or {}; det=usage.get('completion_tokens_details') or {}
    return {
        'raw_text':normalize_content(msg.get('content')),
        'finish_reason':ch.get('finish_reason',''),
        'returned_model':data.get('model',''),
        'returned_provider':data.get('provider',''),
        'latency_ms':latency,
        'prompt_tokens':usage.get('prompt_tokens',np.nan),
        'completion_tokens':usage.get('completion_tokens',np.nan),
        'reasoning_tokens':det.get('reasoning_tokens',usage.get('reasoning_tokens',np.nan)),
        'total_tokens':usage.get('total_tokens',np.nan),
        'reported_cost':usage.get('cost',data.get('cost',np.nan)),
        'response_id':data.get('id','')
    }

def evaluate_unit(stage,key,info,row,rep):
    rec={c:'' for c in RESULT_COLUMNS}
    rec.update({
        'timestamp_utc':utc_now_iso(), 'run_id':RUN_ID, 'stage':stage,
        'evaluator_key':key, 'display_name':info['display_name'],
        'developer':info['developer'], 'family':info['family'], 'access_type':info['access_type'],
        'requested_model':info['model_id'], 'requested_provider':SELECTED_PROVIDERS[key],
        'case_id':row['case_id'], 'repetition':rep,
        'temperature_sent':REQUESTED_TEMPERATURE if SEND_TEMPERATURE else '',
        'max_tokens':MAX_TOKENS, 'api_success':False,
        'json_strict_valid':False, 'json_extractable':False, 'schema_valid':False, 'logical_valid':False,
        'latency_ms':np.nan, 'prompt_tokens':np.nan, 'completion_tokens':np.nan,
        'reasoning_tokens':np.nan, 'total_tokens':np.nan, 'reported_cost':np.nan
    })
    try:
        a=call_openrouter(key,info,row)
        rec.update({
            'api_success':True,'raw_response':a['raw_text'],'returned_model':a['returned_model'],
            'returned_provider':a['returned_provider'],'finish_reason':a['finish_reason'],
            'latency_ms':a['latency_ms'],'prompt_tokens':a['prompt_tokens'],
            'completion_tokens':a['completion_tokens'],'reasoning_tokens':a['reasoning_tokens'],
            'total_tokens':a['total_tokens'],'reported_cost':a['reported_cost'],'response_id':a['response_id']
        })
        if not a['raw_text']:
            rec.update({'error_class':'EmptyModelContent','error_message':'message.content kosong'})
            return rec
        try:
            p,strict,ex=parse_json_object(a['raw_text']); schema,logical,errs=validate_prediction(p)
            rec.update({
                'extracted_json':ex,'json_strict_valid':strict,'json_extractable':True,
                'schema_valid':schema,'logical_valid':logical,'validation_errors':' | '.join(errs),
                'pred_error_type':p.get('error_type',''),'pred_severity':p.get('severity',''),
                'pred_verdict':p.get('verdict',''),'pred_reason':p.get('reason','')
            })
        except Exception as exc:
            rec.update({'error_class':type(exc).__name__,'error_message':str(exc)[:3000]})
    except Exception as exc:
        rec.update({'error_class':type(exc).__name__,'error_message':str(exc)[:3000]})
    return rec

def append_result(rec):
    pd.DataFrame([{c:rec.get(c,'') for c in RESULT_COLUMNS}]).to_csv(
        ATTEMPTS_PATH,mode='a',header=not ATTEMPTS_PATH.exists(),index=False,encoding='utf-8-sig'
    )

def load_attempts():
    if not ATTEMPTS_PATH.exists():return pd.DataFrame(columns=RESULT_COLUMNS)
    d=pd.read_csv(ATTEMPTS_PATH,dtype=str,keep_default_na=False,encoding='utf-8-sig'); d['_o']=np.arange(len(d))
    return d.sort_values('_o').drop_duplicates(['run_id','evaluator_key','case_id','repetition'],keep='last').drop(columns='_o').reset_index(drop=True)

def completed_keys():
    d=load_attempts()
    if d.empty:return set()
    d=d[d['api_success'].astype(str).str.lower().eq('true')]
    return {(r['evaluator_key'],r['case_id'],int(r['repetition'])) for _,r in d.iterrows() if str(r['repetition']).isdigit()}

## 12. Smoke test — 1 kasus × 6 model

In [ ]:
if RUN_SMOKE_TEST:
    row=df_blind[df_blind['case_id'].eq(SMOKE_CASE_ID)].iloc[0]
    smoke=[]
    for key,info in MODELS.items():
        print('\nSmoke',key,info['display_name'])
        rec=evaluate_unit('smoke',key,info,row,0); smoke.append(rec)
        if not rec['api_success']:raise RuntimeError(f'Smoke API gagal {key}: {rec["error_class"]} {rec["error_message"]}')
        print('provider=',rec['returned_provider'] or rec['requested_provider'],'json_strict=',rec['json_strict_valid'],'schema=',rec['schema_valid'],'logical=',rec['logical_valid'])
    df_smoke=pd.DataFrame(smoke); display(df_smoke[['evaluator_key','display_name','api_success','json_strict_valid','schema_valid','logical_valid','returned_provider','latency_ms']])
else: print('Smoke test dilewati.')


Smoke E1 GPT-5.2
provider= OpenAI json_strict= True schema= True logical= True

Smoke E2 Claude Sonnet 4.6
provider= Anthropic json_strict= False schema= True logical= True

Smoke E3 Gemini 3.1 Pro Preview
provider= Google AI Studio json_strict= True schema= True logical= True

Smoke E4 Llama 4 Maverick
provider= DeepInfra json_strict= True schema= True logical= True

Smoke E5 Qwen3-32B
provider= DeepInfra json_strict= True schema= True logical= True

Smoke E6 DeepSeek V4 Pro
provider= Novita json_strict= True schema= True logical= True


,evaluator_key,display_name,api_success,json_strict_valid,schema_valid,logical_valid,returned_provider,latency_ms
0,E1,GPT-5.2,True,True,True,True,OpenAI,8148.604625
1,E2,Claude Sonnet 4.6,True,False,True,True,Anthropic,5191.582218
2,E3,Gemini 3.1 Pro Preview,True,True,True,True,Google AI Studio,3621.085509
3,E4,Llama 4 Maverick,True,True,True,True,DeepInfra,3037.413478
4,E5,Qwen3-32B,True,True,True,True,DeepInfra,4395.808709
5,E6,DeepSeek V4 Pro,True,True,True,True,Novita,14846.268890


In [ ]:
display(df_smoke[['evaluator_key','display_name','completion_tokens','reasoning_tokens','total_tokens','reported_cost','latency_ms']])

,evaluator_key,display_name,completion_tokens,reasoning_tokens,total_tokens,reported_cost,latency_ms
0,E1,GPT-5.2,400,307,1205,0.007009,8148.604625
1,E2,Claude Sonnet 4.6,200,0,1299,0.006297,5191.582218
2,E3,Gemini 3.1 Pro Preview,226,171,1068,0.004396,3621.085509
3,E4,Llama 4 Maverick,81,0,856,0.000220,3037.413478
4,E5,Qwen3-32B,437,388,1383,0.000198,4395.808709
5,E6,DeepSeek V4 Pro,962,892,1829,0.003260,14846.268890


## 13. Runner benchmark + Tahap A

In [ ]:
def run_units(stage,keys,reps):
    done=completed_keys(); units=[]
    for _,row in df_blind.iterrows():
        for rep in reps:
            local=keys.copy(); random.Random(SEED+sum(map(ord,str(row['case_id'])))+rep).shuffle(local)
            for key in local:
                if (key,row['case_id'],rep) not in done:units.append((key,row,rep))
    print(stage,'pending',len(units))
    for key,row,rep in tqdm(units,desc=stage):
        rec=evaluate_unit(stage,key,MODELS[key],row,rep); append_result(rec)
        if not rec['api_success']:raise RuntimeError(f'API benchmark gagal {key}/{row["case_id"]}/rep{rep}: {rec["error_class"]} {rec["error_message"]}')
        time.sleep(SLEEP_BETWEEN_CALLS_SECONDS)

if RUN_STAGE_A:run_units('stage_a_capability',list(MODELS),[1])

stage_a_capability pending 318


stage_a_capability:   0%|          | 0/318 [00:00<?, ?it/s]

## 14. Scoring Tahap A + shortlist 4

In [ ]:
INVALID='__invalid__'
def scored(d):
    x=d.copy()
    for c in ['api_success','json_strict_valid','json_extractable','schema_valid','logical_valid']:
        x[c]=x[c].astype(str).str.lower().eq('true')
    for c in ['repetition','latency_ms','prompt_tokens','completion_tokens','reasoning_tokens','total_tokens','reported_cost']:
        x[c]=pd.to_numeric(x[c],errors='coerce')
    x=x.merge(df_gold[['case_id','gold_error_type','gold_severity','gold_verdict']],on='case_id',how='left',validate='many_to_one')
    ok=x['logical_valid']
    x['score_error_type']=np.where(ok,x['pred_error_type'],INVALID)
    x['score_severity']=np.where(ok,x['pred_severity'],INVALID)
    x['score_verdict']=np.where(ok,x['pred_verdict'],INVALID)
    return x

def mf1(y,p,labels):
    return float(f1_score(y,p,labels=labels,average='macro',zero_division=0))
def metrics(g):
    return {
        'n':len(g),
        'macro_f1_error_type':mf1(g['gold_error_type'],g['score_error_type'],ERROR_TYPES),
        'accuracy_error_type':float(accuracy_score(g['gold_error_type'],g['score_error_type'])),
        'macro_f1_severity':mf1(g['gold_severity'],g['score_severity'],SEVERITIES),
        'accuracy_severity':float(accuracy_score(g['gold_severity'],g['score_severity'])),
        'macro_f1_verdict':mf1(g['gold_verdict'],g['score_verdict'],VERDICTS),
        'accuracy_verdict':float(accuracy_score(g['gold_verdict'],g['score_verdict'])),
        'joint_exact_match':float((
            g['score_error_type'].eq(g['gold_error_type']) &
            g['score_severity'].eq(g['gold_severity']) &
            g['score_verdict'].eq(g['gold_verdict'])
        ).mean()),
        'api_success_rate':float(g['api_success'].mean()),
        'json_strict_valid_rate':float(g['json_strict_valid'].mean()),
        'schema_valid_rate':float(g['schema_valid'].mean()),
        'logical_valid_rate':float(g['logical_valid'].mean()),
        'mean_latency_ms':float(g['latency_ms'].mean()),
        'total_reported_cost':float(g['reported_cost'].fillna(0).sum())
    }

a=scored(load_attempts())
a=a[a['repetition'].eq(1)&a['evaluator_key'].isin(MODELS)]
if len(a)!=STAGE_A_EXPECTED:
    raise RuntimeError(f'Stage A belum lengkap: {len(a)}/{STAGE_A_EXPECTED}')
rows=[]
for key,info in MODELS.items():
    g=a[a['evaluator_key'].eq(key)]
    rows.append({
        'evaluator_key':key,'display_name':info['display_name'],'developer':info['developer'],
        'family':info['family'],'access_type':info['access_type'],
        'model_id':info['model_id'],'provider':SELECTED_PROVIDERS[key],**metrics(g)
    })
df_stage_a=pd.DataFrame(rows).sort_values(
    ['macro_f1_error_type','joint_exact_match','logical_valid_rate','schema_valid_rate'],
    ascending=[False,False,False,False],kind='stable'
).reset_index(drop=True)
df_stage_a['capability_rank']=np.arange(1,len(df_stage_a)+1)
df_stage_a.to_csv(STAGE_A_PATH,index=False,encoding='utf-8-sig')
df_shortlist=df_stage_a.head(SHORTLIST_SIZE).copy()
SHORTLIST_KEYS=df_shortlist['evaluator_key'].tolist()
df_shortlist.to_csv(SHORTLIST_PATH,index=False,encoding='utf-8-sig')
display(df_stage_a[[
    'capability_rank','evaluator_key','display_name','developer','family','access_type',
    'macro_f1_error_type','accuracy_error_type','macro_f1_severity','macro_f1_verdict',
    'joint_exact_match','json_strict_valid_rate','schema_valid_rate','logical_valid_rate','total_reported_cost'
]].round(4))
print('SHORTLIST:',SHORTLIST_KEYS)

,capability_rank,evaluator_key,display_name,developer,family,access_type,macro_f1_error_type,accuracy_error_type,macro_f1_severity,macro_f1_verdict,joint_exact_match,json_strict_valid_rate,schema_valid_rate,logical_valid_rate,total_reported_cost
0,1,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,0.9051,0.9057,0.6885,0.7756,0.6981,1.0000,1.0000,1.0000,0.5183
1,2,E1,GPT-5.2,OpenAI,GPT,Proprietary,0.7208,0.7547,0.8111,0.8515,0.6792,1.0000,1.0000,1.0000,0.2605
2,3,E2,Claude Sonnet 4.6,Anthropic,Claude,Proprietary,0.6745,0.6792,0.6115,0.7114,0.5849,0.3962,1.0000,1.0000,0.3148
3,4,E6,DeepSeek V4 Pro,DeepSeek,DeepSeek,Open-weight,0.6274,0.5849,0.5744,0.5931,0.5472,0.6792,0.6792,0.6792,0.1837
4,5,E5,Qwen3-32B,Qwen/Alibaba,Qwen,Open-weight,0.4447,0.4717,0.5890,0.5832,0.4528,1.0000,1.0000,1.0000,0.0119
5,6,E4,Llama 4 Maverick,Meta,Llama,Open-weight,0.4194,0.4528,0.5601,0.7114,0.4528,1.0000,1.0000,1.0000,0.0119


SHORTLIST: ['E3', 'E1', 'E2', 'E6']


## 15. Tahap B — repetition 2 dan 3 shortlist

In [ ]:
if RUN_STAGE_B:run_units('stage_b_stability',SHORTLIST_KEYS,[2,3])

stage_b_stability pending 424


stage_b_stability:   0%|          | 0/424 [00:00<?, ?it/s]

## 16. Performance 3 repetition + stability

In [ ]:
a=scored(load_attempts())
s=a[a['evaluator_key'].isin(SHORTLIST_KEYS)&a['repetition'].isin(range(1,TOTAL_REPETITIONS_SHORTLIST+1))].copy()
STAGE_B_TOTAL_EXPECTED = SHORTLIST_SIZE * EXPECTED_CASES * TOTAL_REPETITIONS_SHORTLIST
if len(s)!=STAGE_B_TOTAL_EXPECTED:
    raise RuntimeError(f'Stage B belum lengkap: {len(s)}/{STAGE_B_TOTAL_EXPECTED}')

def entropy_norm(vals):
    c=Counter(vals); n=sum(c.values())
    if n<=1 or len(c)<=1:return 0.0
    p=np.array([v/n for v in c.values()],float)
    return float(-np.sum(p*np.log(p))/math.log(n))
def pair_agree(vals):
    return float(np.mean([x==y for x,y in itertools.combinations(vals,2)]))
def majority(vals):
    lab,n=Counter(vals).most_common(1)[0]
    return lab if n>=2 else '__no_majority__'

rep=[]
for key in SHORTLIST_KEYS:
    for r in range(1,TOTAL_REPETITIONS_SHORTLIST+1):
        info=MODELS[key]
        rep.append({
            'evaluator_key':key,'display_name':info['display_name'],'developer':info['developer'],
            'family':info['family'],'access_type':info['access_type'],'repetition':r,
            **metrics(s[(s['evaluator_key']==key)&(s['repetition']==r)])
        })
df_rep=pd.DataFrame(rep)

case=[]
for key in SHORTLIST_KEYS:
    for cid,g in s[s['evaluator_key']==key].groupby('case_id',sort=False):
        g=g.sort_values('repetition')
        e=g['score_error_type'].tolist(); sev=g['score_severity'].tolist(); v=g['score_verdict'].tolist()
        case.append({
            'evaluator_key':key,'case_id':cid,
            'error_pairwise_agreement':pair_agree(e),
            'severity_pairwise_agreement':pair_agree(sev),
            'verdict_pairwise_agreement':pair_agree(v),
            'error_unanimous':float(len(set(e))==1),
            'severity_unanimous':float(len(set(sev))==1),
            'verdict_unanimous':float(len(set(v))==1),
            'error_normalized_entropy':entropy_norm(e),
            'severity_normalized_entropy':entropy_norm(sev),
            'verdict_normalized_entropy':entropy_norm(v),
            'majority_error_type':majority(e),
            'majority_severity':majority(sev),
            'majority_verdict':majority(v)
        })
df_case=pd.DataFrame(case)

final=[]
for key in SHORTLIST_KEYS:
    info=MODELS[key]
    rg=df_rep[df_rep['evaluator_key']==key]
    cg=df_case[df_case['evaluator_key']==key]
    maj=cg.merge(df_gold[['case_id','gold_error_type','gold_severity','gold_verdict']],on='case_id',validate='one_to_one')
    final.append({
        'evaluator_key':key,'display_name':info['display_name'],'developer':info['developer'],
        'family':info['family'],'access_type':info['access_type'],
        'model_id':info['model_id'],'provider':SELECTED_PROVIDERS[key],
        'mean_macro_f1_error_type':rg['macro_f1_error_type'].mean(),
        'std_macro_f1_error_type':rg['macro_f1_error_type'].std(ddof=0),
        'majority_macro_f1_error_type':mf1(maj['gold_error_type'],maj['majority_error_type'],ERROR_TYPES),
        'mean_joint_exact_match':rg['joint_exact_match'].mean(),
        'error_pairwise_agreement':cg['error_pairwise_agreement'].mean(),
        'error_unanimous_rate':cg['error_unanimous'].mean(),
        'mean_error_normalized_entropy':cg['error_normalized_entropy'].mean(),
        'mean_schema_valid_rate':rg['schema_valid_rate'].mean(),
        'mean_logical_valid_rate':rg['logical_valid_rate'].mean(),
        'total_reported_cost_3reps':s.loc[s['evaluator_key']==key,'reported_cost'].fillna(0).sum()
    })
df_stage_b=pd.DataFrame(final).sort_values(
    ['mean_macro_f1_error_type','error_pairwise_agreement','mean_joint_exact_match','mean_logical_valid_rate'],
    ascending=[False,False,False,False],kind='stable'
).reset_index(drop=True)
df_stage_b['stability_rank']=np.arange(1,len(df_stage_b)+1)
df_stage_b.to_csv(STAGE_B_PATH,index=False,encoding='utf-8-sig')
display(df_stage_b.round(4))

,evaluator_key,display_name,developer,family,access_type,model_id,provider,mean_macro_f1_error_type,std_macro_f1_error_type,majority_macro_f1_error_type,mean_joint_exact_match,error_pairwise_agreement,error_unanimous_rate,mean_error_normalized_entropy,mean_schema_valid_rate,mean_logical_valid_rate,total_reported_cost_3reps,stability_rank
0,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,google/gemini-3.1-pro-preview,google-ai-studio,0.9121,0.0231,0.9051,0.7107,0.9497,0.9245,0.0437,1.0000,1.0000,1.4826,1
1,E1,GPT-5.2,OpenAI,GPT,Proprietary,openai/gpt-5.2,openai,0.6936,0.0209,0.7090,0.6101,0.8742,0.8113,0.1093,1.0000,1.0000,0.7815,2
2,E2,Claude Sonnet 4.6,Anthropic,Claude,Proprietary,anthropic/claude-sonnet-4.6,anthropic,0.6631,0.0253,0.6739,0.5849,0.8931,0.8491,0.0954,0.9937,0.9937,0.9657,3
3,E6,DeepSeek V4 Pro,DeepSeek,DeepSeek,Open-weight,deepseek/deepseek-v4-pro,novita/fp8,0.6349,0.0373,0.5827,0.5723,0.7484,0.6415,0.2236,0.7044,0.7044,0.4965,4


## 17. Dukungan pemilihan panel 3 evaluator

Panel final **tidak dipaksa** harus berisi campuran proprietary/open-weight tertentu. Tiga evaluator utama direkomendasikan dari ranking capability + stability.

Sebagai diagnostik tambahan, bagian ini menghitung:
- jumlah developer/family unik;
- komposisi access type;
- **inter-model agreement** berdasarkan majority label per kasus setelah 3 repetisi.

Inter-model agreement **bukan confidence** dan bukan kriteria yang berdiri sendiri. Agreement lebih rendah dapat menunjukkan komplementaritas, tetapi juga dapat muncul karena salah satu model buruk. Karena itu interpretasikan selalu bersama performa dan kestabilan.

In [ ]:
df_top3=df_stage_b.head(3).copy(); df_top3['selection_type']='top3_performance_stability_primary'; df_top3.to_csv(PANEL_TOP3_PATH,index=False,encoding='utf-8-sig')
err_wide=df_case.pivot(index='case_id',columns='evaluator_key',values='majority_error_type'); sev_wide=df_case.pivot(index='case_id',columns='evaluator_key',values='majority_severity'); verdict_wide=df_case.pivot(index='case_id',columns='evaluator_key',values='majority_verdict')
def cross_model_agreement(combo,wide):
    vals=[]
    for a,b in itertools.combinations(combo,2):
        q=wide[[a,b]].dropna()
        if len(q): vals.append(float(q[a].eq(q[b]).mean()))
    return float(np.mean(vals)) if vals else np.nan
comb=[]
for combo in itertools.combinations(df_stage_b['evaluator_key'].tolist(),3):
    m=df_stage_b[df_stage_b['evaluator_key'].isin(combo)]; access_types=sorted(m['access_type'].unique().tolist())
    comb.append({'panel':' + '.join(combo),'members':' | '.join(m['display_name']),'unique_developers':int(m['developer'].nunique()),'unique_families':int(m['family'].nunique()),'access_types':' + '.join(access_types),'n_access_types':len(access_types),'mean_macro_f1_error_type':float(m['mean_macro_f1_error_type'].mean()),'min_macro_f1_error_type':float(m['mean_macro_f1_error_type'].min()),'mean_self_error_agreement':float(m['error_pairwise_agreement'].mean()),'mean_joint_exact_match':float(m['mean_joint_exact_match'].mean()),'inter_model_error_agreement':cross_model_agreement(combo,err_wide),'inter_model_verdict_agreement':cross_model_agreement(combo,verdict_wide)})
df_combo=pd.DataFrame(comb).sort_values(['mean_macro_f1_error_type','min_macro_f1_error_type','mean_self_error_agreement','mean_joint_exact_match'],ascending=[False,False,False,False],kind='stable').reset_index(drop=True); df_combo['combination_rank']=np.arange(1,len(df_combo)+1); df_combo.to_csv(PANEL_COMB_PATH,index=False,encoding='utf-8-sig')
TOP3_KEYS=df_top3['evaluator_key'].tolist()
def panel_majority(values):
    vals=[v for v in values if pd.notna(v) and str(v)!='']
    if not vals: return '__no_majority__',0
    lab,n=Counter(vals).most_common(1)[0]; return (lab,n) if n>=2 else ('__no_majority__',n)
panel_rows=[]
for cid in df_gold['case_id']:
    pe,ne=panel_majority([err_wide.loc[cid,k] for k in TOP3_KEYS]); ps,ns=panel_majority([sev_wide.loc[cid,k] for k in TOP3_KEYS]); pv,nv=panel_majority([verdict_wide.loc[cid,k] for k in TOP3_KEYS]); g=df_gold[df_gold['case_id'].eq(cid)].iloc[0]
    panel_rows.append({'case_id':cid,'panel_members':' | '.join(TOP3_KEYS),'panel_error_type':pe,'panel_error_agreement_count':ne,'panel_severity':ps,'panel_severity_agreement_count':ns,'panel_verdict':pv,'panel_verdict_agreement_count':nv,'gold_error_type':g['gold_error_type'],'gold_severity':g['gold_severity'],'gold_verdict':g['gold_verdict'],'panel_error_correct':int(pe==g['gold_error_type']),'panel_severity_correct':int(ps==g['gold_severity']),'panel_verdict_correct':int(pv==g['gold_verdict'])})
df_panel_case=pd.DataFrame(panel_rows); df_panel_case.to_csv(PANEL_CASE_PATH,index=False,encoding='utf-8-sig')
print('Rekomendasi Top-3 berdasarkan performance + stability:'); display(df_top3[['evaluator_key','display_name','developer','family','access_type','mean_macro_f1_error_type','std_macro_f1_error_type','error_pairwise_agreement','mean_joint_exact_match','mean_logical_valid_rate']].round(4))
print('Diagnostik seluruh kombinasi panel:'); display(df_combo.round(4))
print('Diagnostik majority panel Top-3:'); print('Panel members:',TOP3_KEYS); print('Verdict accuracy:',round(float(df_panel_case['panel_verdict_correct'].mean()),4)); print('Error-type majority available:',f"{int(df_panel_case['panel_error_type'].ne('__no_majority__').sum())}/{len(df_panel_case)}"); display(df_panel_case.head())


Rekomendasi Top-3 berdasarkan performance + stability:


,evaluator_key,display_name,developer,family,access_type,mean_macro_f1_error_type,std_macro_f1_error_type,error_pairwise_agreement,mean_joint_exact_match,mean_logical_valid_rate
0,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,0.9121,0.0231,0.9497,0.7107,1.0000
1,E1,GPT-5.2,OpenAI,GPT,Proprietary,0.6936,0.0209,0.8742,0.6101,1.0000
2,E2,Claude Sonnet 4.6,Anthropic,Claude,Proprietary,0.6631,0.0253,0.8931,0.5849,0.9937


Diagnostik seluruh kombinasi panel:


,panel,members,unique_developers,unique_families,access_types,n_access_types,mean_macro_f1_error_type,min_macro_f1_error_type,mean_self_error_agreement,mean_joint_exact_match,inter_model_error_agreement,inter_model_verdict_agreement,combination_rank
0,E3 + E1 + E2,Gemini 3.1 Pro Preview | GPT-5.2 | Claude Sonn...,3,3,Proprietary,1,0.7563,0.6631,0.9057,0.6352,0.7358,0.9245,1
1,E3 + E1 + E6,Gemini 3.1 Pro Preview | GPT-5.2 | DeepSeek V4...,3,3,Open-weight + Proprietary,2,0.7469,0.6349,0.8574,0.6310,0.6164,0.7044,2
2,E3 + E2 + E6,Gemini 3.1 Pro Preview | Claude Sonnet 4.6 | D...,3,3,Open-weight + Proprietary,2,0.7367,0.6349,0.8637,0.6226,0.6226,0.7170,3
3,E1 + E2 + E6,GPT-5.2 | Claude Sonnet 4.6 | DeepSeek V4 Pro,3,3,Open-weight + Proprietary,2,0.6639,0.6349,0.8386,0.5891,0.6038,0.6981,4


Diagnostik majority panel Top-3:
Panel members: ['E3', 'E1', 'E2']
Verdict accuracy: 0.8491
Error-type majority available: 51/53


,case_id,panel_members,panel_error_type,panel_error_agreement_count,panel_severity,panel_severity_agreement_count,panel_verdict,panel_verdict_agreement_count,gold_error_type,gold_severity,gold_verdict,panel_error_correct,panel_severity_correct,panel_verdict_correct
0,C001,E3 | E1 | E2,correct,2,none,2,layak,2,correct,none,layak,1,1,1
1,C002,E3 | E1 | E2,untranslated,3,major,2,perlu_ditinjau,3,untranslated,major,perlu_ditinjau,1,1,1
2,C003,E3 | E1 | E2,correct,2,none,2,layak,2,untranslated,major,perlu_ditinjau,0,0,0
3,C004,E3 | E1 | E2,missing_information,3,major,3,perlu_ditinjau,3,missing_information,major,perlu_ditinjau,1,1,1
4,C005,E3 | E1 | E2,hallucination,3,critical,3,perlu_ditinjau,3,hallucination,critical,perlu_ditinjau,1,1,1


## 18. Error analysis + completeness + biaya

In [ ]:
err=[]
for key in SHORTLIST_KEYS:
    g=s[s['evaluator_key']==key]
    for lab in ERROR_TYPES:
        q=g[g['gold_error_type']==lab]
        if len(q): err.append({'evaluator_key':key,'display_name':MODELS[key]['display_name'],'developer':MODELS[key]['developer'],'family':MODELS[key]['family'],'access_type':MODELS[key]['access_type'],'gold_error_type':lab,'n':len(q),'accuracy_within_gold_category':q['score_error_type'].eq(lab).mean(),'most_common_prediction':q['score_error_type'].value_counts().index[0]})
df_error=pd.DataFrame(err)
complete=s.groupby(['evaluator_key','display_name','developer','family','access_type','requested_provider'],as_index=False).agg(observations=('case_id','size'),unique_cases=('case_id','nunique'),api_success_rate=('api_success','mean'),schema_valid_rate=('schema_valid','mean'),logical_valid_rate=('logical_valid','mean'),total_reported_cost=('reported_cost','sum'),mean_latency_ms=('latency_ms','mean'))
CONFUSION_LABELS=ERROR_TYPES+[INVALID]; confusion_tables={}; classification_tables={}
for key in SHORTLIST_KEYS:
    g=s[s['evaluator_key'].eq(key)]; cm=confusion_matrix(g['gold_error_type'],g['score_error_type'],labels=CONFUSION_LABELS); confusion_tables[key]=pd.DataFrame(cm,index=[f'gold_{x}' for x in CONFUSION_LABELS],columns=[f'pred_{x}' for x in CONFUSION_LABELS]); report=classification_report(g['gold_error_type'],g['score_error_type'],labels=ERROR_TYPES,output_dict=True,zero_division=0); classification_tables[key]=pd.DataFrame(report).T
print('Completeness/cost'); display(complete.round(4)); print('Error analysis'); display(df_error.round(4))
for key in SHORTLIST_KEYS: print(f'Confusion matrix {key} - {MODELS[key]["display_name"]}'); display(confusion_tables[key])


Completeness/cost


,evaluator_key,display_name,developer,family,access_type,requested_provider,observations,unique_cases,api_success_rate,schema_valid_rate,logical_valid_rate,total_reported_cost,mean_latency_ms
0,E1,GPT-5.2,OpenAI,GPT,Proprietary,openai,159,53,1.0,1.0000,1.0000,0.7815,4810.5430
1,E2,Claude Sonnet 4.6,Anthropic,Claude,Proprietary,anthropic,159,53,1.0,0.9937,0.9937,0.9657,4236.5882
2,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,google-ai-studio,159,53,1.0,1.0000,1.0000,1.4826,6792.2483
3,E6,DeepSeek V4 Pro,DeepSeek,DeepSeek,Open-weight,novita/fp8,159,53,1.0,0.7044,0.7044,0.4965,21425.0768


Error analysis


,evaluator_key,display_name,developer,family,access_type,gold_error_type,n,accuracy_within_gold_category,most_common_prediction
0,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,correct,18,0.6667,correct
1,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,meaning_shift,18,0.8889,meaning_shift
2,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,missing_information,18,0.9444,missing_information
3,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,added_information,18,1.0000,added_information
4,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,hallucination,18,1.0000,hallucination
5,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,untranslated,18,1.0000,untranslated
6,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,empty_output,15,1.0000,empty_output
7,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,fluency_issue,18,0.7222,fluency_issue
8,E3,Gemini 3.1 Pro Preview,Google,Gemini,Proprietary,source_leakage,18,1.0000,source_leakage
9,E1,GPT-5.2,OpenAI,GPT,Proprietary,correct,18,0.6667,correct


Confusion matrix E3 - Gemini 3.1 Pro Preview


,pred_correct,pred_meaning_shift,pred_missing_information,pred_added_information,pred_hallucination,pred_untranslated,pred_empty_output,pred_fluency_issue,pred_source_leakage,pred___invalid__
gold_correct,12,4,0,0,2,0,0,0,0,0
gold_meaning_shift,2,16,0,0,0,0,0,0,0,0
gold_missing_information,0,1,17,0,0,0,0,0,0,0
gold_added_information,0,0,0,18,0,0,0,0,0,0
gold_hallucination,0,0,0,0,18,0,0,0,0,0
gold_untranslated,0,0,0,0,0,18,0,0,0,0
gold_empty_output,0,0,0,0,0,0,15,0,0,0
gold_fluency_issue,2,0,0,3,0,0,0,13,0,0
gold_source_leakage,0,0,0,0,0,0,0,0,18,0
gold___invalid__,0,0,0,0,0,0,0,0,0,0


... (2 outputs left out)

Confusion matrix E2 - Claude Sonnet 4.6


,pred_correct,pred_meaning_shift,pred_missing_information,pred_added_information,pred_hallucination,pred_untranslated,pred_empty_output,pred_fluency_issue,pred_source_leakage,pred___invalid__
gold_correct,10,7,0,1,0,0,0,0,0,0
gold_meaning_shift,3,15,0,0,0,0,0,0,0,0
gold_missing_information,0,2,16,0,0,0,0,0,0,0
gold_added_information,0,1,0,10,6,0,0,0,0,1
gold_hallucination,0,1,0,0,17,0,0,0,0,0
gold_untranslated,3,0,0,0,0,15,0,0,0,0
gold_empty_output,0,0,0,0,0,0,15,0,0,0
gold_fluency_issue,3,9,0,0,0,3,0,3,0,0
gold_source_leakage,0,0,0,0,0,12,0,0,6,0
gold___invalid__,0,0,0,0,0,0,0,0,0,0


Confusion matrix E6 - DeepSeek V4 Pro


,pred_correct,pred_meaning_shift,pred_missing_information,pred_added_information,pred_hallucination,pred_untranslated,pred_empty_output,pred_fluency_issue,pred_source_leakage,pred___invalid__
gold_correct,6,6,0,0,0,0,0,0,0,6
gold_meaning_shift,3,13,0,0,2,0,0,0,0,0
gold_missing_information,0,0,17,0,0,0,0,0,0,1
gold_added_information,0,0,0,5,0,0,0,0,0,13
gold_hallucination,0,0,0,0,16,0,0,0,0,2
gold_untranslated,0,0,0,0,0,18,0,0,0,0
gold_empty_output,0,0,0,0,0,0,15,0,0,0
gold_fluency_issue,0,0,0,1,0,1,0,2,0,14
gold_source_leakage,0,0,0,0,0,4,0,0,3,11
gold___invalid__,0,0,0,0,0,0,0,0,0,0


## 19. Ekspor laporan final

In [ ]:
a_all=scored(load_attempts()); a_all.to_csv(RUN_DIR/'benchmark_attempts_scored.csv',index=False,encoding='utf-8-sig'); df_rep.to_csv(RUN_DIR/'repetition_metrics.csv',index=False,encoding='utf-8-sig'); df_case.to_csv(RUN_DIR/'case_stability.csv',index=False,encoding='utf-8-sig'); df_error.to_csv(RUN_DIR/'error_analysis.csv',index=False,encoding='utf-8-sig'); complete.to_csv(RUN_DIR/'completeness_cost_provider.csv',index=False,encoding='utf-8-sig')
with pd.ExcelWriter(REPORT_XLSX,engine='openpyxl') as w:
    df_gold_full.to_excel(w,'Benchmark_Gold',index=False); df_validation_sync.to_excel(w,'Validation_Sync',index=False); df_preflight.to_excel(w,'Preflight',index=False); df_evidence.to_excel(w,'Shared_Evidence',index=False); df_stage_a.to_excel(w,'StageA_Capability',index=False); df_shortlist.to_excel(w,'Shortlist_4',index=False); df_rep.to_excel(w,'Repetition_Metrics',index=False); df_stage_b.to_excel(w,'StageB_Stability',index=False); df_combo.to_excel(w,'Panel_Diagnostics',index=False); df_top3.to_excel(w,'Top3_Final',index=False); df_panel_case.to_excel(w,'Top3_Case_Majority',index=False); df_error.to_excel(w,'Error_Analysis',index=False); complete.to_excel(w,'Completeness_Cost',index=False)
    for key in SHORTLIST_KEYS: confusion_tables[key].to_excel(w,f'CM_{key}'); classification_tables[key].to_excel(w,f'Report_{key}')
print('SELESAI'); print('RUN_DIR:',RUN_DIR); print('Stage A:',STAGE_A_PATH); print('Shortlist:',SHORTLIST_PATH); print('Stage B:',STAGE_B_PATH); print('Top3 final recommendation:',PANEL_TOP3_PATH); print('Top3 panel case majority:',PANEL_CASE_PATH); print('Panel diagnostics:',PANEL_COMB_PATH); print('Excel:',REPORT_XLSX)


SELESAI
RUN_DIR: /content/drive/MyDrive/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_evaluator_openrouter_FINAL_V5/run_372a575a9fa21f91
Stage A: /content/drive/MyDrive/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_evaluator_openrouter_FINAL_V5/run_372a575a9fa21f91/stage_a_capability_summary.csv
Shortlist: /content/drive/MyDrive/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_evaluator_openrouter_FINAL_V5/run_372a575a9fa21f91/stage_a_shortlist_4.csv
Stage B: /content/drive/MyDrive/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_evaluator_openrouter_FINAL_V5/run_372a575a9fa21f91/stage_b_performance_stability_summary.csv
Top3 final recommendation: /content/drive/MyDrive/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_evaluator_openrouter_FINAL_V5/run_372a575a9fa21f91/panel_top3_final_recommendation.csv
Top3 panel case majority: /content/drive/MyDrive/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_evaluator_openrouter_FINAL_V5/run_372a575a9fa21f91/panel_top3_case_predicti

/tmp/ipykernel_544/3824303974.py:3: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df_gold_full.to_excel(w,'Benchmark_Gold',index=False); df_validation_sync.to_excel(w,'Validation_Sync',index=False); df_preflight.to_excel(w,'Preflight',index=False); df_evidence.to_excel(w,'Shared_Evidence',index=False); df_stage_a.to_excel(w,'StageA_Capability',index=False); df_shortlist.to_excel(w,'Shortlist_4',index=False); df_rep.to_excel(w,'Repetition_Metrics',index=False); df_stage_b.to_excel(w,'StageB_Stability',index=False); df_combo.to_excel(w,'Panel_Diagnostics',index=False); df_top3.to_excel(w,'Top3_Final',index=False); df_panel_case.to_excel(w,'Top3_Case_Majority',index=False); df_error.to_excel(w,'Error_Analysis',index=False); complete.to_excel(w,'Completeness_Cost',index=False)
/tmp/ipykernel_544/3824303974.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the 

## 20. Interpretasi hasil

- **Macro-F1 `error_type` adalah metrik utama**.
- Severity, verdict, joint exact match, validitas JSON, konsistensi logis, biaya, dan latensi adalah metrik pendukung.
- Tahap A menyaring Top-4 dari 6 kandidat; Tahap B menilai performa dan stabilitasnya pada tiga pengulangan.
- **Top-3 final** diambil dari performa + stabilitas tanpa memaksa campuran proprietary/open-weight.
- Agreement antar-model diagnostik, bukan correctness.
- Verdict panel selalu majority 2/3 atau 3/3; error_type/severity dapat `__no_majority__` pada pola 1-1-1.
- Evaluator tetap **reference-free**; reference/gold hanya untuk scoring offline.
- Test set tidak dibaca notebook benchmark evaluator.
